# Medical Text Classification Training (Pro)
This notebook trains a high-accuracy binary classifier using large-scale real-world datasets:
1. **Medical QA** (`icliniq_medical_qa_cleaned.csv` + `HealthCareMagic-100k.json`) -> Label 1
2. **Real Smalltalk** (`Small_talk_Intent.csv`) -> Label 0
3. **General News** (`AG_NEWS.csv`) -> Label 0
4. **Formal Medical** (`train.dat`) -> Label 1

In [4]:
import pandas as pd
import pickle
import os
import nltk
import json
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Setup NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [5]:
def transform_text(text):
    text = str(text).lower()
    text = nltk.word_tokenize(text)
    y = [i for i in text if i.isalnum()]
    text = [i for i in y if i not in stop_words and i not in punctuation]
    return " ".join([ps.stem(i) for i in text])

## 1. Load and Balance Data

In [6]:
print("Loading datasets...")

# 1. Formal Medical (train.dat)
with open('train.dat', 'r', encoding='utf-8', errors='ignore') as f:
    formal_df = pd.DataFrame(f.readlines(), columns=['text'])
formal_df['label'] = 1

# 2. Medical QA (HealthCareMagic)
with open('HealthCareMagic-100k.json', 'r') as f:
    hcm_df = pd.DataFrame(json.load(f))
hcm_df = hcm_df[['instruction']].rename(columns={'instruction': 'text'})
hcm_df['label'] = 1

# 3. Medical QA (iCliniq)
icliniq_df = pd.read_csv('icliniq_medical_qa_cleaned.csv')
icliniq_df = icliniq_df[['Question']].rename(columns={'Question': 'text'})
icliniq_df['label'] = 1

# 4. Small Talk
small_talk_df = pd.read_csv('Small_talk_Intent.csv')
small_talk_df = small_talk_df[['Utterances']].rename(columns={'Utterances': 'text'})
small_talk_df['label'] = 0

# 5. General News (AG NEWS)
news_df = pd.read_csv('AG_NEWS.csv')
news_df['text'] = news_df['Title'] + " " + news_df['Description']
news_df = news_df[['text']]
news_df['label'] = 0

# Balance the classes (20k Medical, 20k Non-Medical)
medical_combined = pd.concat([
    formal_df.sample(min(5000, len(formal_df)), random_state=42),
    hcm_df.sample(7500, random_state=42),
    icliniq_df.sample(7500, random_state=42)
])

non_medical_combined = pd.concat([
    small_talk_df, # All 2.5k small talk
    news_df.sample(min(17500, len(news_df)), random_state=42)
])

df = pd.concat([medical_combined, non_medical_combined], ignore_index=True)
print(f"Total samples: {len(df)}")
print(df['label'].value_counts())

Loading datasets...
Total samples: 40047
label
0    20047
1    20000
Name: count, dtype: int64


In [7]:
print("Processing text...")
df['transformed_text'] = df['text'].astype(str).apply(transform_text)

print("Vectorizing and Training...")
tfidf = TfidfVectorizer(max_features=30000) # Increased features for more data
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label'].values

model = MultinomialNB()
model.fit(X, y)

with open('vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('model.pkl', 'wb') as f: pickle.dump(model, f)
print("Retraining complete with advanced datasets!")

Processing text...
Vectorizing and Training...
Retraining complete with advanced datasets!
